# SWYNEX Internship – Task 2
## Data Cleaning and Basic Analysis

**Dataset:** *Escherichia coli* str. K-12 substr. MG1655 complete genome  
**NCBI RefSeq accession:** NC_000913.3  
**Source:** NCBI Nucleotide / RefSeq

This notebook downloads the selected reference genome from NCBI, cleans the FASTA sequence, and calculates basic sequence statistics and quality checks.

In [ ]:
import re
from urllib.request import urlopen

ACCESSION = 'NC_000913.3'
FASTA_URL = (
    'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
    '?db=nuccore&id=NC_000913.3&rettype=fasta&retmode=text'
)

print('Dataset:', ACCESSION)
print('Source:', 'NCBI RefSeq')

### 1. Download and clean the FASTA sequence
The header line is separated from the nucleotide sequence. Whitespace and unexpected characters are removed. The cleaned sequence is converted to uppercase.

In [ ]:
raw = urlopen(FASTA_URL, timeout=60).read().decode('utf-8')
lines = raw.strip().splitlines()
header = lines[0]
sequence = ''.join(lines[1:]).upper()
sequence = re.sub(r'\s+', '', sequence)

print(header)
print('Cleaned sequence length:', len(sequence), 'bp')

### 2. Basic sequence statistics
We calculate genome length, nucleotide counts, GC%, AT%, and the number/percentage of ambiguous bases (N).

In [ ]:
from collections import Counter

counts = Counter(sequence)
length = len(sequence)
A = counts.get('A', 0)
T = counts.get('T', 0)
G = counts.get('G', 0)
C = counts.get('C', 0)
N = counts.get('N', 0)
ambiguous = length - (A + T + G + C)

gc_pct = ((G + C) / length) * 100
at_pct = ((A + T) / length) * 100
n_pct = (N / length) * 100

print(f'Genome length: {length:,} bp')
print(f'A: {A:,}')
print(f'T: {T:,}')
print(f'G: {G:,}')
print(f'C: {C:,}')
print(f'N: {N:,}')
print(f'GC content: {gc_pct:.2f}%')
print(f'AT content: {at_pct:.2f}%')
print(f'Ambiguous bases: {ambiguous:,} ({ambiguous/length*100:.4f}%)')

### 3. Quality checks
For a complete reference genome, we check whether the sequence contains only standard DNA bases (A, T, G, C) and whether ambiguous bases are present.

In [ ]:
allowed = set('ATGCN')
unexpected = sorted(set(sequence) - allowed)

print('Unexpected characters:', unexpected)
print('Contains ambiguous N bases:', N > 0)
print('Sequence passes character check:', len(unexpected) == 0)

### 4. Save a summary table
The summary can be committed to GitHub as a CSV file for reproducibility.

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    'Metric': ['Accession', 'Genome length (bp)', 'A bases', 'T bases', 'G bases', 'C bases', 'N bases', 'GC content (%)', 'AT content (%)', 'Ambiguous bases (%)'],
    'Value': [ACCESSION, length, A, T, G, C, N, round(gc_pct, 2), round(at_pct, 2), round(ambiguous/length*100, 4)]
})
display(summary)
summary.to_csv('task2_sequence_summary.csv', index=False)
print('Saved: task2_sequence_summary.csv')

## Interpretation

The selected *E. coli* K-12 MG1655 reference genome is a complete circular chromosome of 4,641,652 bp. Published genome summaries report approximately 50.79% GC content and 4,285 coding sequences. The analysis above independently recalculates sequence-level statistics from the downloaded FASTA file. A low/zero ambiguous-base count is expected for a complete reference sequence and provides a simple quality check.